# ToDo

# Подготовка

## Настройка графики

In [ ]:
# windows.options(height=5.4, width=7)
oldpar <- par()
par(mar = c(10, 4, 1, 2), "xpd" = FALSE)
par(cex.axis = 2, cex.axis = 2, cex.lab = 2)
options(repr.plot.height = 9, repr.plot.width = 12)
options(warn = -1)

## Библиотеки

In [ ]:
options(java.parameters = "-Xmx4096m")

require(readxl, quietly = TRUE, warn.conflicts = FALSE)

require(vcd, quietly = TRUE, warn.conflicts = FALSE)
require(coin, quietly = TRUE, warn.conflicts = FALSE)
# independence_test
require(agricolae, quietly = TRUE, warn.conflicts = FALSE)
# HSD.test
require(pgirmess, quietly = TRUE, warn.conflicts = FALSE)
# kruskalmc
require(nortest, quietly = TRUE, warn.conflicts = FALSE)
# for normality test in case of N>5000. ad.test -- Anderson-Darling normality test
require(RcmdrMisc, quietly = TRUE, warn.conflicts = FALSE)
# numSumm

require(beeswarm, quietly = TRUE, warn.conflicts = FALSE)
require(lattice, quietly = TRUE, warn.conflicts = FALSE)
require(mosaic, quietly = TRUE, warn.conflicts = FALSE)
require(ggplot2, quietly = TRUE, warn.conflicts = FALSE)
require(ggpubr, quietly = TRUE, warn.conflicts = FALSE)
# ggqqplot
# require(ggExtra, quietly = TRUE, warn.conflicts = FALSE);
# require(gridExtra, quietly = TRUE, warn.conflicts = FALSE);
# require(ggfortify, quietly = TRUE, warn.conflicts = FALSE);
# require(ggalluvial, quietly = TRUE);                            # flow diagramm
# require(hrbrthemes, quietly = TRUE, warn.conflicts = FALSE);    # ggparcoord
# require(GGally, quietly = TRUE, warn.conflicts = FALSE);        # ggparcoord
# require(viridis, quietly = TRUE, warn.conflicts = FALSE);       # ggparcoord


require(rstatix, quietly = TRUE);                               # identify_outliers

require(dplyr, quietly = TRUE, warn.conflicts = FALSE)
require(tidyr, quietly = TRUE, warn.conflicts = FALSE)
require(tidycmprsk, quietly = TRUE, warn.conflicts = FALSE)
# require(tidyverse, quietly = TRUE, warn.conflicts = FALSE);

require(IRdisplay, quietly = TRUE, warn.conflicts = FALSE)
# display
# require(repr, quietly = TRUE, warn.conflicts = FALSE);

# require(TeachingDemos, quietly = TRUE, warn.conflicts = FALSE);
# require(Rmisc, quietly = TRUE, warn.conflicts = FALSE);
# require(ranger, quietly = TRUE, warn.conflicts = FALSE);
# require(Epi, quietly = TRUE, warn.conflicts = FALSE);
# require(car, quietly = TRUE, warn.conflicts = FALSE);
# require(exactRankTests, quietly = TRUE, warn.conflicts = FALSE);
# require(abind, quietly = TRUE, warn.conflicts = FALSE);
# require(mstate, quietly = TRUE, warn.conflicts = FALSE);
# require(gnm, quietly = TRUE, warn.conflicts = FALSE);
# require(multcomp, quietly = TRUE, warn.conflicts = FALSE);
# require(scales, quietly = TRUE, warn.conflicts = FALSE);
# require(Rcmdr, quietly = TRUE, warn.conflicts = FALSE);
# require(tigerstats, quietly = TRUE, warn.conflicts = FALSE);
# require(fpp3e, quietly = TRUE, warn.conflicts = FALSE);
# require(tsibble, quietly = TRUE, warn.conflicts = FALSE);
# require(lubridate, quietly = TRUE, warn.conflicts = FALSE);
# require(cowplot, quietly = TRUE, warn.conflicts = FALSE);
# require(smplot2, quietly = TRUE, warn.conflicts = FALSE);
# require(biostat3, quietly = TRUE, warn.conflicts = FALSE);
# require(data.table, quietly = TRUE, warn.conflicts = FALSE);

## Функции

In [ ]:
build_sankey <- function(data, group, parameter, group_colors) {
  for (groupID in parameter) {
    groupSize <- nlevels(data[[groupID]])
    group_colors1 <- group_colors[1:groupSize]
    for (i in 2:nlevels(data[[group]])) {
      tmp_tab <- table(data[data[[group]] == levels(data[[group]])[i - 1], groupID], data[data[[group]] == levels(data[[group]])[i], groupID])
      colnames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i], 2, "left", "0"), colnames(tmp_tab), sep = "_")
      rownames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i - 1], 2, "left", "0"), rownames(tmp_tab), sep = "_")
      if (i == 2) {
        out_tab <- as.data.table(tmp_tab)
        colnames(out_tab) <- c("source", "target", "value")
      } else {
        out_tab <- rbind(out_tab, as.data.table(tmp_tab), use.names = FALSE)
      }
    }
    out_tab <- cbind(as.data.frame(out_tab), group = rep(group_colors1, ceiling(dim(out_tab)[1] / length(group_colors1)))[1:dim(out_tab)[1]])

    links <- data.frame(out_tab[out_tab$value != 0, ])
    links <- links[order(links$source), ]

    nodes <- data.frame(name = unique(c(as.character(links$source), as.character(links$target))))
    unique_status <- sort(unique(substring(nodes$name, unlist(gregexpr(".[0-9]+$", gsub("\\+", ".", nodes$name))) + 1)))
    nodes$group <- group_colors1[match(gsub(".*\\.([0-9]+)$", "\\1", gsub("\\+", ".", nodes$name)), unique_status)]
    nodes <- nodes[order(nodes$name), ]
    rowTotals <- unique(rbind(aggregate(links$value, by = list(links$source), FUN = sum), aggregate(links$value, by = list(links$target), FUN = sum)))
    names(rowTotals) <- c("group", "x")
    nodes$rowtotal <- rowTotals$x
    nodes <- transform(nodes, rowpc = ave(rowtotal, substring(nodes$name, 1, 3), FUN = prop.table) * 100)

    links$IDsource <- match(links$source, nodes$name) - 1
    links$IDtarget <- match(links$target, nodes$name) - 1

    color <- sprintf(
      "d3.scaleOrdinal().domain([%s]).range([%s]);",
      paste0("'", paste(nodes$name, collapse = "", ""), "'"),
      paste0("'", paste(nodes$group, collapse = "", ""), "'")
    )

    #    links$group = substring(links$group, 2)
    #    nodes$group = substring(nodes$group, 2)

    sn <- sankeyNetwork(
      Links = links, Nodes = nodes,
      Source = "IDsource", Target = "IDtarget", Value = "value",
      NodeID = "name", LinkGroup = "source", NodeGroup = "name",
      units = "cases",
      sinksRight = FALSE, nodeWidth = 30, nodePadding = 30,
      colourScale = color,
      fontSize = 14, fontFamily = "Arial"
    )
    print(sn)
  }
}

metrics <- c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "s110", "m_s110")
# group_colors = c("#f6e1f7ff", "#ecaad6FF", "#de6bb8ff", "#cc2e97ff", "#9d40a3FF", "#a7157FF", "#82003EFF", "#5e002dFF")
group_colors <- c("#0084ffff", "#44bec7ff", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157fff", "#82003eff", "#5e002dff", "#000000ff")
metrics <- c("d330")
# build_sankey(preml, "time", metrics, group_colors)


build_sankeyP <- function(data, group, parameter, group_colors) {
  for (groupID in parameter) {
    groupSize <- nlevels(data[[groupID]])
    group_colors1 <- group_colors[1:groupSize]
    for (i in 2:nlevels(data[[group]])) {
      tmp_tab <- table(data[data[[group]] == levels(data[[group]])[i - 1], groupID], data[data[[group]] == levels(data[[group]])[i], groupID])
      colnames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i], 2, "left", "0"), colnames(tmp_tab), sep = "_")
      rownames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i - 1], 2, "left", "0"), rownames(tmp_tab), sep = "_")
      if (i == 2) {
        out_tab <- as.data.table(tmp_tab)
        colnames(out_tab) <- c("source", "target", "value")
      } else {
        out_tab <- rbind(out_tab, as.data.table(tmp_tab), use.names = FALSE)
      }
    }
    out_tab <- cbind(as.data.frame(out_tab), group = rep(group_colors1, ceiling(dim(out_tab)[1] / length(group_colors1)))[1:dim(out_tab)[1]])

    links <- data.frame(out_tab[out_tab$value != 0, ])
    links <- links[order(links$source), ]

    nodes <- data.frame(name = unique(c(as.character(links$source), as.character(links$target))))
    unique_status <- sort(unique(substring(nodes$name, unlist(gregexpr(".[0-9]+$", gsub("\\+", ".", nodes$name))) + 1)))
    nodes$group <- group_colors1[match(gsub(".*\\.([0-9]+)$", "\\1", gsub("\\+", ".", nodes$name)), unique_status)]
    nodes <- nodes[order(nodes$name), ]
    rowTotals <- unique(rbind(aggregate(links$value, by = list(links$source), FUN = sum), aggregate(links$value, by = list(links$target), FUN = sum)))
    names(rowTotals) <- c("group", "x")
    nodes$rowtotal <- rowTotals$x
    nodes <- transform(nodes, rowpc = ave(rowtotal, substring(nodes$name, 1, 3), FUN = prop.table) * 100)

    links$IDsource <- match(links$source, nodes$name) - 1
    links$IDtarget <- match(links$target, nodes$name) - 1

    link <- list(
      source = links$IDsource,
      target = links$IDtarget,
      value = links$value,
      color = links$group
    )
    node <- list(
      label = sprintf("%i (%2.1f%%)", nodes$rowtotal, nodes$rowpc),
      color = nodes$group,
      pad = 10,
      line = list(
        color = "black",
        width = 1
      ),
      thickness = 30
    )
    domain <- list(
      x = c(0, 1),
      y = c(0, 1)
    )

    p <- plot_ly(
      link = link,
      node = node,
      domain = domain,
      type = "sankey",
      orientation = "h",
      valueformat = " .0f "
      # , valuesuffix = " случаи"
      , alpha = 1,
      height = 650,
      width = 850
    )

    p <- config(
      p,
      scrollZoom = TRUE,
      responsive = TRUE
      # , staticPlot = TRUE
      # , displayModeBar = TRUE
      , displaylogo = FALSE
    ) %>% layout(
      autosize = TRUE,
      title = groupID,
      font = list(size = 14),
      margin = list(
        l = 50,
        r = 50,
        b = 0,
        t = 100
        # , pad = 4
        # , automargin = TRUE
      ),
      xaxis = list(title = "Sepal Length (cm)"),
      legend = list(x = 1, y = 0.5)
    )
    print(p)
  }
}

# group_colors = c("#0084ff99", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("#0084ff99", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("rgba(0,255,255,0.4)", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("#0084ffff", "#44bec7FF", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157ff", "#82003eff", "#5e002dff")
metrics <- c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "s110", "m_s110")
group_colors <- c("#0084ffff", "#44bec7ff", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157fff", "#82003eff", "#5e002dff", "#000000ff")
metrics <- c("b750")


# build_sankeyP(lorl, "time", metrics, group_colors)


build_sankey_strataG <- function(data, group, strata, parameter, group_colors, save_multiple, save_file) {
  if (save_multiple) {
    pdf(sprintf("%s.pdf", strata), onefile = TRUE, paper = "a4r", width = 20, height = 14, encoding = "KOI8-R")
  }
  for (groupID in parameter) {
    in_data <- data[!is.na(data[[groupID]]), ]
    p <- ggplot(
      in_data,
      aes(
        x = in_data[[group]],
        stratum = in_data[[groupID]],
        alluvium = in_data[["uid"]],
        fill = in_data[[groupID]]
      )
      # , width = 1980
      # , height = 1024
    ) +
      facet_wrap(
        in_data[[strata]],
        scales = "free",
        strip.position = "top",
        drop = TRUE
      ) +
      geom_flow(
        # stat = "alluvium"
        # , lode.guidance = "frontback"
        ,
        width = 0.27,
        color = "darkgray",
        curve_type = "arctangent" # "linear", "cubic", "quintic", "sine", "arctangent", and "sigmoid". "xspline"
      ) +
      geom_stratum(alpha = .5) +
      geom_text(
        stat = "flow",
        aes(
          label = sprintf(" %i ", after_stat(n)),
          hjust = (after_stat(flow) == "to"),
          vjust = (after_stat(flow) == "to")
        ),
        size = 4
      ) +
      geom_text(
        stat = "stratum",
        aes(
          label = sprintf("%i\n(%3.1f%%)", after_stat(n), after_stat(prop * 100))
          # , hjust = as.integer(after_stat(stratum)) - 2
        ),
        hjust = 0,
        size = 5,
        nudge_x = 0.2
      ) +
      # expand_limits(x = 0, y = 0) +
      xlab(
        group
      ) +
      labs(fill = groupID, size = 5) +
      theme(
        legend.position = "bottom",
        panel.background = element_rect(fill = NA),
        panel.border = element_blank(),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        strip.text = element_text(size = 14), ,
        axis.ticks.y = element_blank(),
        axis.text.y = element_blank(),
        axis.text.x = element_text(size = 12),
        axis.title.x = element_text(size = 14),
        axis.line.x = element_line(size = 1, colour = "black", linetype = 1),
        legend.text = element_text(size = 14),
        legend.title = element_blank(),
        panel.spacing.x = unit(0.5, "lines"),
        text = element_text(family = "Arial Narrow")
      ) +
      scale_fill_manual(values = group_colors) +
      ggtitle(groupID)
    if (save_file) {
      ggsave(sprintf("%s_%s.png", strata, groupID), plot = p, width = 36, height = 24, unit = "cm", dpi = "print")
    }
    print(p)
  }
  if (save_multiple) {
    dev.off()
  }
}


build_sankey_strata <- function(data, group, strata, parameter, group_colors) {
  strata_data <- list()
  for (i in 1:nlevels(data[[strata]])) {
    tmp_data <- subset(data, data[[strata]] == levels(data[[strata]])[i])
    strata_data[[i]] <- tmp_data
  }
  for (groupID in parameter) {
    for (i in seq_along(strata_data)) {
      print(paste(levels(data[[strata]])[i], ", ", groupID))
      build_sankey(strata_data[[i]], group, groupID, group_colors)
    }
  }
}

metrics <- c("m_s110")
# metrics = c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "m_s110")
group_colors <- c("#0084ff", "#44bec7", "#ffc300", "#fa3c4c", "#d696bb", "#a7157f", "#82003e", "#5e002d", "#000000")

# build_sankey_strataG(preml, "time", "Абилитация", metrics, group_colors, FALSE, TRUE)

In [ ]:
library(fmsb, quietly = TRUE, warn.conflicts = FALSE)
library(plotrix, quietly = TRUE, warn.conflicts = FALSE)
library(circlize, quietly = TRUE, warn.conflicts = FALSE)

extract_value <- function(x) {
  return(as.numeric(gsub(".+[.+]{1}([-0-9]+)", "\\1", gsub("-", ".-", x))))
}

draw.circle2 <- function(x, y, radius, nv = 100, border = NULL, col = NA, lty = 1, density = NULL, angle = 45, lwd = 1) {
  xylim <- par("usr")
  plotdim <- par("pin")
  ymult <- getYmult()
  angle.inc <- 2 * pi / nv
  angles <- seq(0, 2 * pi - angle.inc, by = angle.inc)
  if (length(col) < length(radius)) {
    col <- rep(col, length.out = length(radius))
  }
  for (circle in 1:length(radius)) {
    xv <- sin(angles) * radius[circle] + x
    yv <- cos(angles) * radius[circle] * ymult + y
    polygon(xv, yv,
      border = border, col = col[circle], lty = lty,
      density = density, angle = angle, lwd = lwd
    )
  }
  invisible(list(x = xv, y = yv))
}

build_radar <- function(
  data, id, metrics, colors_border, labels = "", radar_title = "", with_legend = TRUE, with_nulls = TRUE, axis_min = NULL, axis_max = NULL, axis_label_col = "dark gray",
  negative_fill = "black", negative_transparency = 0.5,
  arcs = FALSE, arc_colors
) {
  b <- data[data$id == id, ]
  # 	a = data.frame(lapply(b[, c(metrics)], as.numeric))
  a <- data.frame(lapply(b[, c(metrics)], extract_value))
  if (!with_nulls) {
    a[is.na(a)] <- 0
  }

  if (is.null(axis_min)) {
    min_value <- min(a, na.rm = TRUE)
  } else {
    min_value <- axis_min
  }
  if (is.null(axis_max)) {
    max_value <- max(a, na.rm = TRUE)
  } else {
    max_value <- axis_max
  }
  a <- rbind(rep(min_value, length(metrics)), a)
  a <- rbind(rep(max_value, length(metrics)), a)
  rownames(a) <- c("1", "2", paste(b$time, "мес."))

  negative <- FALSE
  if (min_value < 0) {
    negative <- TRUE
    shift <- abs(min_value)
    axis_labels <- as.character(seq(min_value, max_value))
    a <- a + shift
    min_value <- 0
    max_value <- max_value + shift
  } else {
    axis_labels <- NULL
  }

  colors_fill <- alpha(colors_border, 0.15)

  if (length(labels) == 1 && labels[1] == "") {
    radarchart(a,
      axistype = 1, pcol = colors_border, pfcol = colors_fill,
      seg = a[1, 1], plwd = 4, plty = 1, cglcol = "grey", cglty = 1, axislabcol = axis_label_col,
      vlcex = 1, cex.main = 1.5, centerzero = FALSE, caxislabels = seq(0, 5, 1), title = radar_title
    )
  } else {
    radarchart(a,
      axistype = 1, pcol = colors_border, pfcol = colors_fill,
      seg = a[1, 1], plwd = 4, plty = 1, cglcol = "grey", cglty = 1, axislabcol = axis_label_col,
      vlabels = labels, vlcex = 1, cex.main = 1.5, centerzero = FALSE, caxislabels = seq(0, 5, 1), title = radar_title
    )
  }

  angle_step <- 360 / length(metrics)
  par("xpd" = TRUE)
  if (negative) {
    fill_radius <- (shift + 1) * 0.98 / (max_value + min_value + 1)
    draw.circle2(0, 0, fill_radius, nv = length(metrics), angle = 45, col = alpha(negative_fill, negative_transparency), border = negative_fill, lwd = 1)
    # filledcircle(r1 = fill_radius, mid = c(0, 0), dr = dr, col = alpha(negative_fill, negative_transparency))
  }

  if (arcs) {
    if (!exists("arc_colors")) {
      arc_colors <- seq(1, 5, 1)
    }

    draw.arc(0, 0, 1.4, deg1 = angle_step * 5, deg2 = angle_step * 13 - angle_step / 4, lwd = 3, col = arc_colors[1])
    draw.radial.line(1.2, 1.4, center = c(0, 0), deg = angle_step * 5, lwd = 3, col = arc_colors[1])
    draw.radial.line(1.2, 1.4, center = c(0, 0), deg = angle_step * 13 - angle_step / 4, lwd = 3, col = arc_colors[1])

    draw.arc(0, 0, 1.4, deg1 = angle_step * 13, deg2 = angle_step * 17 - angle_step / 4, lwd = 3, col = arc_colors[2])
    draw.radial.line(1.2, 1.4, center = c(0, 0), deg = angle_step * 13, lwd = 3, col = arc_colors[2])
    draw.radial.line(1.2, 1.4, center = c(0, 0), deg = angle_step * 17 - angle_step / 4, lwd = 3, col = arc_colors[2])

    draw.arc(0, 0, 1.4, deg1 = angle_step * 17, deg2 = angle_step * 21 - angle_step / 4, lwd = 3, col = arc_colors[3])
    draw.radial.line(1.2, 1.4, center = c(0, 0), deg = angle_step * 17, lwd = 3, col = arc_colors[3])
    draw.radial.line(1.2, 1.4, center = c(0, 0), deg = angle_step * 21 - angle_step / 4, lwd = 3, col = arc_colors[3])

    draw.arc(0, 0, 1.4, deg1 = angle_step * 21, deg2 = angle_step * 27 - angle_step / 4, lwd = 3, col = arc_colors[4])
    draw.radial.line(1.2, 1.4, center = c(0, 0), deg = angle_step * 21, lwd = 3, col = arc_colors[4])
    draw.radial.line(1.2, 1.4, center = c(0, 0), deg = angle_step * 27 - angle_step / 4, lwd = 3, col = arc_colors[4])
  }

  par("xpd" = FALSE)

  if (with_legend) {
    legend(x = 1.3, y = 1.3, legend = rownames(a[-c(1, 2), ]), bty = "n", pch = 20, col = colors_border, text.col = "black", cex = 1.2, pt.cex = 3)
  }
}

## Данные

### Загрузка

In [ ]:
# sessionInfo()
# options(encoding = "UTF-8")
lor <- read_excel("C:\\Analysis\\OTOLARING\\Nidelko\\lor.xlsx", sheet = "данные")
lor <- as.data.frame(lor)

### Преобразование

#### Отбор данных

In [ ]:
# SNOT 22
lor <- lor %>%
  select(c(
    "uid", "группа", "id", "возраст", "пол",
    "длительность заболевания", "удаление полипов в анамнезе", "хронический ринит", "АР", "БА"
    # , 'искривление носовой перегородки', 'повторная аллотрансплантация', 'оперированные пазухи', 'верхнечелюстная', 'этмоидальная', 'фронтальная', 'сфеноидальная', 'FESS'
    # , 'вазотомия нижних носовых раковин', 'операция на перегородке носа', 'распространенность полипозного процесса КТ'
    # , 'лобные пазухи', 'клиновидные пазухи', 'решетчатый лабиринт передний', 'решетчатый лабиринт задний', 'верхнечелюстные пазухи', 'остиомеатальный комплекс', 'шкала Лунд-Маккей'
    , "сморкание.0", "сморкание.3",
    "чихание.0", "чихание.3",
    "насморк.0", "насморк.3",
    "заложенность носа.0", "заложенность носа.3",
    "аносмия и потеря вкуса.0", "аносмия и потеря вкуса.3",
    "кашель.0", "кашель.3",
    "затекание слизи по задней стенке глотки.0", "затекание слизи по задней стенке глотки.3",
    "густые выделения из носа.0", "густые выделения из носа.3",
    "заложенность в ушах.0", "заложенность в ушах.3",
    "головокружение.0", "головокружение.3",
    "боль в ушах.0", "боль в ушах.3",
    "боль в лице.0", "боль в лице.3",
    "трудно заснуть.0", "трудно заснуть.3",
    "ночные пробуждения.0", "ночные пробуждения.3",
    "плохой сон ночью.0", "плохой сон ночью.3",
    "просыпаюсь уставшим.0", "просыпаюсь уставшим.3",
    "хроническа усталость.0", "хроническа усталость.3",
    "снижение активности.0", "снижение активности.3",
    "снижение концентарации внимания.0", "снижение концентарации внимания.3",
    "подавленность.0", "подавленность.3",
    "уныние.0", "уныние.3",
    "рассеянность.0", "рассеянность.3"
    # , 'отделяемое', 'корки', 'отек', 'бальная оценка'
  ))
lor <- as.data.frame(lor)

#### Контрасты

In [ ]:
lor$группа <- factor(lor$группа, c("ОГ1", "ОГ2", "КГ"))
lor$пол <- factor(lor$пол)

lor$"хронический ринит" <- factor(lor$"хронический ринит")
lor$"АР" <- factor(lor$"АР")
lor$"БА" <- factor(lor$"БА")

# lor$"искривление носовой перегородки" = factor(lor$"искривление носовой перегородки")
# lor$"повторная аллотрансплантация" = factor(lor$"повторная аллотрансплантация")
# lor$"верхнечелюстная" = factor(lor$"верхнечелюстная", c("нет", "односторонняя", "двустороняя"))
# lor$"этмоидальная" = factor(lor$"этмоидальная", c("нет", "односторонняя", "двустороняя"))
# lor$"фронтальная" = factor(lor$"фронтальная", c("нет", "односторонняя", "двустороняя"))
# lor$"сфеноидальная" = factor(lor$"сфеноидальная", c("нет", "односторонняя", "двустороняя"))
# lor$"вазотомия нижних носовых раковин" = factor(lor$"вазотомия нижних носовых раковин")
# lor$"операция на перегородке носа" = factor(lor$"операция на перегородке носа")
# lor$FESS = factor(lor$FESS)

# lor$"распространенность полипозного процесса КТ" = factor(lor$"распространенность полипозного процесса КТ")
# lor$"лобные пазухи" = factor(lor$"лобные пазухи")
# lor$"клиновидные пазухи" = factor(lor$"клиновидные пазухи")
# lor$"решетчатый лабиринт передний" = factor(lor$"решетчатый лабиринт передний")
# lor$"решетчатый лабиринт задний" = factor(lor$"решетчатый лабиринт задний")
# lor$"верхнечелюстные пазухи" = factor(lor$"верхнечелюстные пазухи")
# lor$"остиомеатальный комплекс" = factor(lor$"остиомеатальный комплекс")

# lor$"отделяемое" = factor(lor$"отделяемое")
# lor$"корки" = factor(lor$"корки")
# lor$"отек" = factor(lor$"отек")

#### Суммарные

In [ ]:
lor$"ринологические симптомы.0" <- lor$"сморкание.0" + lor$"чихание.0" + lor$"насморк.0" + lor$"заложенность носа.0" + lor$"аносмия и потеря вкуса.0" + lor$"кашель.0" + lor$"затекание слизи по задней стенке глотки.0" + lor$"густые выделения из носа.0"
lor$"ринологические симптомы.3" <- lor$"сморкание.3" + lor$"чихание.3" + lor$"насморк.3" + lor$"заложенность носа.3" + lor$"аносмия и потеря вкуса.3" + lor$"кашель.3" + lor$"затекание слизи по задней стенке глотки.3" + lor$"густые выделения из носа.3"

lor$"ухо или лицо.0" <- lor$"заложенность в ушах.0" + lor$"головокружение.0" + lor$"боль в ушах.0" + lor$"боль в лице.0"
lor$"ухо или лицо.3" <- lor$"заложенность в ушах.3" + lor$"головокружение.3" + lor$"боль в ушах.3" + lor$"боль в лице.3"

lor$"качество сна.0" <- lor$"трудно заснуть.0" + lor$"ночные пробуждения.0" + lor$"плохой сон ночью.0" + lor$"просыпаюсь уставшим.0"
lor$"качество сна.3" <- lor$"трудно заснуть.3" + lor$"ночные пробуждения.3" + lor$"плохой сон ночью.3" + lor$"просыпаюсь уставшим.3"

lor$"психические функции.0" <- lor$"хроническа усталость.0" + lor$"снижение активности.0" + lor$"снижение концентарации внимания.0" + lor$"подавленность.0" + lor$"уныние.0" + lor$"рассеянность.0"
lor$"психические функции.3" <- lor$"хроническа усталость.3" + lor$"снижение активности.3" + lor$"снижение концентарации внимания.3" + lor$"подавленность.3" + lor$"уныние.3" + lor$"рассеянность.3"

lor$SNOT22.0 <- lor$"ринологические симптомы.0" + lor$"ухо или лицо.0" + lor$"качество сна.0" + lor$"психические функции.0"
lor$SNOT22.3 <- lor$"ринологические симптомы.3" + lor$"ухо или лицо.3" + lor$"качество сна.3" + lor$"психические функции.3"

# lor$"ринологические симптомы.2" = lor$"сморкание.2" + lor$"чихание.2" + lor$"насморк.2" + lor$"заложенность носа.2" + lor$"аносмия и потеря вкуса.2" + lor$"кашель.2" + lor$"затекание слизи по задней стенке глотки.2" + lor$"густые выделения из носа.2"
# lor$"ухо или лицо.2" = lor$"заложенность в ушах.2" + lor$"головокружение.2" + lor$"боль в ушах.2" + lor$"боль в лице.2"
# lor$"качество сна.2" = lor$"трудно заснуть.2" + lor$"ночные пробуждения.2" + lor$"плохой сон ночью.2" + lor$"просыпаюсь уставшим.2"
# lor$"психические функции.2" = lor$"хроническа усталость.2" + lor$"снижение активности.2" + lor$"снижение концентарации внимания.2" + lor$"подавленность.2" + lor$"уныние.2" + lor$"рассеянность.2"

#### Длинная таблица

In [ ]:
lor <- as.data.frame(lor)
# print(names(lor))
lorl <- reshape(lor,
  direction = "long",
  idvar = "uid",
  sep = ".",
  varying = list(
    11:12, 13:14, 15:16, 17:18, 19:20, 21:22, 23:24, 25:26,
    27:28, 29:30, 31:32, 33:34,
    35:36, 37:38, 39:40, 41:42,
    43:44, 45:46, 47:48, 49:50, 51:52, 53:54,
    55:56, 57:58, 59:60, 61:62,
    63:64
  ),
  v.names = c(
    "сморкание", "чихание", "насморк", "заложенность носа", "аносмия и потеря вкуса", "кашель", "затекание слизи по задней стенке глотки", "густые выделения из носа",
    "заложенность в ушах", "головокружение", "боль в ушах", "боль в лице",
    "трудно заснуть", "ночные пробуждения", "плохой сон ночью", "просыпаюсь уставшим",
    "хроническа усталость", "снижение активности", "снижение концентарации внимания", "подавленность", "уныние", "рассеянность",
    "ринологические симптомы", "ухо или лицо", "качество сна", "психические функции",
    "SNOT22"
  ),
  timevar = "time",
  times = c("до операции", "1 год")
)
lorl$time <- factor(lorl$time, c("до операции", "1 год"))

### Подключение

In [ ]:
try(detach(lor), silent = TRUE)
try(detach(lorl), silent = TRUE)
attach(lorl)

# Общий анализ

In [ ]:
groupping_variable <- "группа"

## сморкание

### Общее

In [ ]:
parname <- "сморкание"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(t.test(tmpset[[parname]] ~ tmpset$time))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox.test(tmpset[[parname]] ~ tmpset$time))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## чихание

### Общее

In [ ]:
parname <- "чихание"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  test_formula <- as.formula(paste0("`", parname, "` ~ time"))
  display(t_test(test_formula, data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## насморк

### Общее

In [ ]:
parname <- "насморк"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  test_formula <- as.formula(paste0("`", parname, "` ~ time"))
  display(t_test(test_formula, data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## заложенность носа

### Общее

In [ ]:
parname <- "заложенность носа"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  test_formula <- as.formula(paste0("`", parname, "` ~ time"))
  display(t_test(test_formula, data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## аносмия и потеря вкуса

### Общее

In [ ]:
parname <- "аносмия и потеря вкуса"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  test_formula <- as.formula(paste0("`", parname, "` ~ time"))
  display(t_test(test_formula, data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## кашель

### Общее

In [ ]:
parname <- "кашель"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## затекание слизи по задней стенке глотки

### Общее

In [ ]:
parname <- "затекание слизи по задней стенке глотки"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## густые выделения из носа

### Общее

In [ ]:
parname <- "густые выделения из носа"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## заложенность в ушах

### Общее

In [ ]:
parname <- "заложенность в ушах"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## головокружение

### Общее

In [ ]:
parname <- "головокружение"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## боль в ушах

### Общее

In [ ]:
parname <- "боль в ушах"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## боль в лице

### Общее

In [ ]:
parname <- "боль в лице"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## трудно заснуть

### Общее

In [ ]:
parname <- "трудно заснуть"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## ночные пробуждения

### Общее

In [ ]:
parname <- "ночные пробуждения"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## плохой сон ночью

### Общее

In [ ]:
parname <- "плохой сон ночью"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## просыпаюсь уставшим

### Общее

In [ ]:
parname <- "просыпаюсь уставшим"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## хроническа усталость

### Общее

In [ ]:
parname <- "хроническа усталость"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## снижение активности

### Общее

In [ ]:
parname <- "снижение активности"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## снижение концентарации внимания

### Общее

In [ ]:
parname <- "снижение концентарации внимания"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## подавленность

### Общее

In [ ]:
parname <- "подавленность"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## уныние

### Общее

In [ ]:
parname <- "уныние"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## рассеянность

### Общее

In [ ]:
parname <- "рассеянность"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## ринологические симптомы

### Общее

In [ ]:
parname <- "ринологические симптомы"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## ухо или лицо

### Общее

In [ ]:
parname <- "ухо или лицо"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## качество сна

### Общее

In [ ]:
parname <- "качество сна"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## психические функции

### Общее

In [ ]:
parname <- "психические функции"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## SNOT22

### Общее

In [ ]:
parname <- "SNOT22"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name)

  p <- ggplot(tmpset, aes(x = tmpset[[parname]], fill = time)) +
    geom_histogram(bins = 10, color = "#e9ecef", alpha = 0.4, position = "dodge") +
    scale_fill_manual(values = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2))) +
    theme_classic() +
    xlab(paste0(parname, ", группа: ", group_name)) +
    ylab("количество") +
    theme(
      axis.title = element_text(size = 16),
      axis.text = element_text(size = 14),
      legend.text = element_text(size = 14),
      legend.title = element_text(size = 14)
    )
  display(p)
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(numSummary(tmpset[[parname]],
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = tmpset$time
  ))
  display(favstats(tmpset[[parname]] ~ tmpset$time))
  display(paste())
  display(paste())
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  display(qqmath(~ tmpset[[parname]] | tmpset$time, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = parname, xlab = group_name))
}

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  beeswarm(tmpset[[parname]] ~ tmpset$time,
    method = "swarm", pch = 16, pwcol = tmpset[["пол"]],
    xlab = paste0(parname, ", группа: ", group_name), ylab = "количество", cex = 1.5
  )
  boxplot(tmpset[[parname]] ~ tmpset$time, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(1.65, -max(tmpset[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)
}

### Тест на нормальность

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

  for (i in 1:nlevels(time)) {
    display(paste0("Время — ", levels(time)[i]))
    ss <- subset(tmpset, tmpset$time == levels(time)[i])
    display(try(print(shapiro.test(ss[[parname]]))))

    display(bartlett.test(tmpset[[parname]] ~ tmpset$time))
    display(paste0(""))
  }
  display(paste0(""))
}

### Сравнение, нормальные распределения

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)

    display(t_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### Сравнение, распределение не нормальное

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  display(paste0("======================== ", group_name, " ========================"))
  tmpset <- lorl %>%
    filter(группа == group_name)


  display(wilcox_test(as.formula(paste0("`", parname, "` ~ time")), data = tmpset, paired = TRUE))
}

### График

In [ ]:
beeswarm(lorl[[parname]] ~ lorl$time + lorl$группа,
  method = "swarm", pch = 16, pwcol = lorl[["пол"]],
  xlab = parname, ylab = "количество", cex = 1.2
)
boxplot(lorl[[parname]] ~ lorl$time + lorl$группа, add = TRUE, varwidth = TRUE, outline = FALSE, col = c(rgb(1, 0.6, 0, 0.2), rgb(0, 0.5, 0.5, 0.2)))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(4, -max(lorl[[parname]]) * 0.15, levels(tmpset$пол), col = 1:nlevels(tmpset$пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

# Радарные диаграммы

In [ ]:
snot22 <- c(
  "сморкание", "чихание", "насморк", "заложенность носа", "аносмия и потеря вкуса", "кашель", "затекание слизи по задней стенке глотки", "густые выделения из носа",
  "заложенность в ушах", "головокружение", "боль в ушах", "боль в лице",
  "трудно заснуть", "ночные пробуждения", "плохой сон ночью", "просыпаюсь уставшим",
  "хроническа усталость", "снижение активности", "снижение концентарации внимания", "подавленность", "уныние", "рассеянность"
)
snot_domains <- c("ринологические симптомы", "ухо или лицо", "качество сна", "психические функции")

snot22_labels <- c(
  "сморкание", "чихание", "насморк", "заложенность\nноса", "аносмия\nи потеря вкуса", "кашель", "затекание слизи\nпо задней\nстенке глотки", "густые выделения\nиз носа",
  "заложенность\nв ушах", "головокружение", "боль\nв ушах", "боль\nв лице",
  "трудно\nзаснуть", "ночные\nпробуждения", "плохой сон\nночью", "просыпаюсь\nуставшим",
  "хроническа\nусталость", "снижение\nактивности", "снижение\nконцентарации\nвнимания", "подавленность", "уныние", "рассеянность"
)
snot_domains_labels <- c("ринологические симптомы", "ухо или лицо", "качество сна", "психические функции")
# 	colors_border = c('#0084ff','#44bec7','#ffc300','#fa3c4c','#d696bb','#a7157f','#82003e','#5e002d','#000000')
colors_border <- c("#ffc300", "#44bec7", "#fa3c4c", "#d696bb", "#a7157f", "#82003e", "#5e002d", "#000000")
colors_domains <- c(5, 7, 6, 8)

## Индивидуальные

СНОТ, абсолютные по группам

In [ ]:
# print(as.data.frame(lorl))
for (group_name in levels(lorl[[groupping_variable]])) {
  # print(group_name)
  tmpset <- lorl %>%
    filter(группа == group_name) %>%
    group_by(uid, id, time, `возраст`, `пол`, `длительность заболевания`, `АР`, `БА`) %>%
    arrange(`АР`, `БА`, `возраст`)
  # print(as.data.frame(tmpset))
  vars <- tmpset %>%
    ungroup() %>%
    select(uid, id, "возраст", "пол", "длительность заболевания", "АР", "БА") %>%
    unique()
  # print(as.data.frame(vars))

  for (i in 1:length(vars$id)) {
    # print(as.data.frame(tmpset[i]))
    build_radar(tmpset, vars$id[i], snot22, colors_border, snot22_labels,
      with_legend = FALSE,
      radar_title = paste0("группа: ", group_name, ", id: ", vars$id[i], ", возраст: ", vars$"возраст"[i], ", пол: ", vars$"пол"[i], ", длительность заболевания: ", vars$"длительность заболевания"[i], ", АР: ", vars$"АР"[i], ", БА: ", vars$"БА")[i],
      arcs = TRUE, arc_colors = colors_domains
    )
  }
}

## Группы

СНОТ, медианы по группам

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name) %>%
    group_by(time) %>%
    summarize(across("сморкание":"SNOT22", ~ median(.x, na.rm = TRUE))) %>%
    mutate(id = 1)

  build_radar(tmpset, 1, snot22, colors_border, snot22_labels, with_legend = FALSE, radar_title = group_name, arcs = TRUE, arc_colors = colors_domains)
}

СНОТ, средние по группам

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name) %>%
    group_by(time) %>%
    summarize(across("сморкание":"SNOT22", ~ round(mean(.x, na.rm = TRUE)))) %>%
    mutate(id = 1)

  build_radar(tmpset, 1, snot22, colors_border, snot22_labels, with_legend = FALSE, radar_title = group_name, arcs = TRUE, arc_colors = colors_domains)
}

## Домены

СНОТ, медианы по группам

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name) %>%
    group_by(time) %>%
    summarize(across("сморкание":"SNOT22", ~ median(.x, na.rm = TRUE))) %>%
    mutate(id = 1)

  build_radar(tmpset, 1, snot_domains, colors_border, snot_domains_labels, with_legend = FALSE, radar_title = group_name, arcs = FALSE)
}

СНОТ, средние по группам

In [ ]:
for (group_name in levels(lorl[[groupping_variable]])) {
  tmpset <- lorl %>%
    filter(группа == group_name) %>%
    group_by(time) %>%
    summarize(across("сморкание":"SNOT22", ~ round(mean(.x, na.rm = TRUE)))) %>%
    mutate(id = 1)

  build_radar(tmpset, 1, snot_domains, colors_border, snot_domains_labels, with_legend = FALSE, radar_title = group_name, arcs = FALSE)
}